In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

# Dataset Generation

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

NUM_BANKS = 200
SEED = 42

OUTPUT_DIR = Path("data")

# Paper: bank assets lie between 100 and 10^10.
MIN_ASSETS = 100.0
MAX_ASSETS = 10_000_000_000.0

# Paper: power-law exponent is selected from [1.5, 5].
POWER_LAW_EXPONENT = 2.5

# Paper: customer-loan recovery rate is common to all banks.
RECOVERY_RATE = 0.5

rng = np.random.default_rng(SEED)

In [3]:
# ============================================================
# 1. GENERATE BANK SIZES
# ============================================================

def generate_bank_sizes(n, exponent, minimum, maximum):
    """
    Generate bank sizes using a truncated power-law distribution.

    Probability density:
        p(x) proportional to x^(-exponent)

    The inverse-CDF method keeps values within the paper's
    specified asset range.
    """

    u = rng.random(n)

    a = 1.0 - exponent

    assets = (
        u * (maximum**a - minimum**a)
        + minimum**a
    ) ** (1.0 / a)

    return assets


In [4]:
# ============================================================
# 2. GENERATE INITIAL BALANCE SHEETS
# ============================================================

def generate_balance_sheets(assets):
    """
    Initial balance sheet:

    Assets:
        A = R + C + B

    Liabilities:
        A = D + L + E

    R = cash reserves
    C = customer loans
    B = interbank assets (loans given)

    D = customer deposits
    L = interbank liabilities (loans received)
    E = equity
    """

    n = len(assets)

    equity_ratio = rng.uniform(0.001, 0.25, n)

    deposit_ratio = rng.uniform(
        0.0,
        1.0 - equity_ratio
    )

    reserve_ratio = rng.uniform(0.0, 0.25, n)

    customer_loan_parameter = rng.uniform(0.0, 1.0, n)

    equity = equity_ratio * assets
    deposits = deposit_ratio * assets
    reserves = reserve_ratio * assets

    # Paper: C_i = max(beta_i * A_i - R_i, 0)
    customer_loans = np.maximum(
        customer_loan_parameter * assets - reserves,
        0.0
    )

    interbank_assets = (
        assets - reserves - customer_loans
    )

    interbank_liabilities = (
        assets - deposits - equity
    )

    return pd.DataFrame({
        "Id": [f"B{i:03d}" for i in range(1, n + 1)],
        "Label": [f"Bank {i:03d}" for i in range(1, n + 1)],

        "initial_assets": assets,

        "initial_reserves": reserves,
        "initial_customer_loans": customer_loans,
        "initial_interbank_assets": interbank_assets,

        "initial_deposits": deposits,
        "initial_interbank_liabilities": interbank_liabilities,
        "initial_equity": equity,

        "initial_equity_ratio": equity_ratio,
        "initial_reserve_ratio": reserve_ratio,
    })


In [ ]:
# ============================================================
# 3. GENERATE DIRECTED SCALE-FREE NETWORK
# ============================================================

def generate_network(bank_assets):
    """
    Generate a directed Albert-Barabasi-style network.

    Larger banks have a higher probability of receiving
    connections because attachment probability depends on
    both network degree and bank size.

    Edge direction:
        lender -> borrower
    """

    n = len(bank_assets)

    G = nx.DiGraph()
    G.add_nodes_from(range(n))

    # Start with a small fully connected directed core.
    initial_nodes = 5

    for i in range(initial_nodes):
        for j in range(initial_nodes):
            if i != j:
                G.add_edge(i, j)

    # Add the remaining banks.
    for new_bank in range(initial_nodes, n):

        existing = np.arange(new_bank)

        # Existing connectivity
        degree = np.array([
            G.in_degree(i) + G.out_degree(i) + 1
            for i in existing
        ], dtype=float)

        # Larger banks receive greater attachment probability
        size = bank_assets[existing]

        # Preferential attachment based on both
        # connectivity and bank size.
        probability = degree * size

        probability /= probability.sum()

        # Number of connections is no longer fixed at 3.
        # Draw a variable number of connections.
        max_edges = len(existing)

        num_edges = rng.integers(
            1,
            min(10, max_edges) + 1
        )

        selected = rng.choice(
            existing,
            size=num_edges,
            replace=False,
            p=probability
        )

        for old_bank in selected:

            # Randomly determine lending direction.
            if rng.random() < 0.5:
                G.add_edge(new_bank, int(old_bank))
            else:
                G.add_edge(int(old_bank), new_bank)

    return G

In [6]:
# ============================================================
# 4. ALLOCATE INTERBANK EXPOSURES
# ============================================================

def allocate_exposures(G, banks):
    """
    Paper's exposure allocation:

        L_ij = Theta_ij * (L_j * B_i) / sum_k(L_k)

    where:

        Theta_ij = 1 if i lends to j
        B_i = initial interbank assets of lender i
        L_j = initial interbank liabilities of borrower j

    Source = lender
    Target = borrower
    """

    total_liabilities = (
        banks["initial_interbank_liabilities"].sum()
    )

    rows = []

    for lender, borrower in G.edges():

        lender_bank = banks.iloc[lender]
        borrower_bank = banks.iloc[borrower]

        B_i = lender_bank["initial_interbank_assets"]
        L_j = borrower_bank["initial_interbank_liabilities"]

        exposure = (
            B_i * L_j / total_liabilities
        )

        if exposure <= 0:
            continue

        rows.append({
            "Source": lender_bank["Id"],
            "Target": borrower_bank["Id"],
            "Type": "Directed",
            "Weight": exposure,

            "relationship": "interbank_loan"
        })

    return pd.DataFrame(rows)


In [7]:
# ============================================================
# 5. REBALANCE BANK BALANCE SHEETS
# ============================================================

def rebalance_balance_sheets(banks, exposures):
    """
    Once network exposures are assigned, the actual interbank
    assets and liabilities are calculated from the edge weights.

    The paper then adjusts the remaining balance-sheet entries
    to restore:

        Assets = Liabilities

    while retaining the original relative composition as far
    as possible.
    """

    banks = banks.copy()

    # Actual loans given: sum of outgoing exposures.
    actual_interbank_assets = (
        exposures.groupby("Source")["Weight"].sum()
    )

    # Actual loans received: sum of incoming exposures.
    actual_interbank_liabilities = (
        exposures.groupby("Target")["Weight"].sum()
    )

    banks["interbank_assets"] = (
        banks["Id"]
        .map(actual_interbank_assets)
        .fillna(0.0)
    )

    banks["interbank_liabilities"] = (
        banks["Id"]
        .map(actual_interbank_liabilities)
        .fillna(0.0)
    )

    # Initial non-interbank assets and liabilities.
    old_non_interbank_assets = (
        banks["initial_reserves"]
        + banks["initial_customer_loans"]
    )

    old_non_interbank_liabilities = (
        banks["initial_deposits"]
        + banks["initial_equity"]
    )

    # Paper's adjusted total-assets rule.
    banks["total_assets"] = np.maximum(
        old_non_interbank_assets
        + banks["interbank_assets"],

        old_non_interbank_liabilities
        + banks["interbank_liabilities"]
    )

    # Remaining amounts after accounting for interbank positions.
    new_non_interbank_assets = (
        banks["total_assets"]
        - banks["interbank_assets"]
    )

    new_non_interbank_liabilities = (
        banks["total_assets"]
        - banks["interbank_liabilities"]
    )

    # Preserve initial reserves/customer-loan proportions.
    asset_scale = (
        new_non_interbank_assets
        / old_non_interbank_assets
    )

    banks["reserves"] = (
        banks["initial_reserves"] * asset_scale
    )

    banks["customer_loans"] = (
        banks["initial_customer_loans"] * asset_scale
    )

    # Preserve initial deposits/equity proportions.
    liability_scale = (
        new_non_interbank_liabilities
        / old_non_interbank_liabilities
    )

    banks["deposits"] = (
        banks["initial_deposits"] * liability_scale
    )

    banks["equity"] = (
        banks["initial_equity"] * liability_scale
    )

    # Ratios useful for subsequent contagion simulations.
    banks["equity_ratio"] = (
        banks["equity"] / banks["total_assets"]
    )

    banks["reserve_ratio"] = (
        banks["reserves"] / banks["total_assets"]
    )

    banks["interbank_asset_ratio"] = (
        banks["interbank_assets"] / banks["total_assets"]
    )

    banks["interbank_liability_ratio"] = (
        banks["interbank_liabilities"]
        / banks["total_assets"]
    )

    return banks

In [8]:
# ============================================================
# 6. ADD NETWORK METRICS FOR GEPHI
# ============================================================

def add_network_metrics(banks, exposures):
    """
    Network metrics are derived attributes, not additional
    assumptions of the paper's balance-sheet model.
    """

    G = nx.DiGraph()

    G.add_nodes_from(banks["Id"])

    for row in exposures.itertuples(index=False):

        G.add_edge(
            row.Source,
            row.Target,
            weight=row.Weight
        )

    banks["out_degree"] = (
        banks["Id"].map(dict(G.out_degree())).fillna(0)
    )

    banks["in_degree"] = (
        banks["Id"].map(dict(G.in_degree())).fillna(0)
    )

    banks["degree"] = (
        banks["in_degree"] + banks["out_degree"]
    )

    banks["weighted_out_degree"] = (
        banks["interbank_assets"]
    )

    banks["weighted_in_degree"] = (
        banks["interbank_liabilities"]
    )

    return banks

In [9]:
# ============================================================
# 7. VALIDATE DATASET
# ============================================================

def validate_dataset(banks, exposures):
    """
    Check financial consistency and graph integrity.
    """

    assert len(banks) == NUM_BANKS

    assert banks["Id"].is_unique

    assert not exposures.empty

    assert not (
        exposures["Source"] == exposures["Target"]
    ).any()

    assert (exposures["Weight"] > 0).all()

    assert exposures[["Source", "Target"]].duplicated().sum() == 0

    assets = (
        banks["reserves"]
        + banks["customer_loans"]
        + banks["interbank_assets"]
    )

    liabilities = (
        banks["deposits"]
        + banks["interbank_liabilities"]
        + banks["equity"]
    )

    assert np.allclose(
        assets,
        liabilities,
        rtol=1e-10,
        atol=1e-6
    ), "Bank balance sheets do not balance."

    assert np.allclose(
        assets,
        banks["total_assets"],
        rtol=1e-10,
        atol=1e-6
    )

    assert np.isclose(
        banks["interbank_assets"].sum(),
        banks["interbank_liabilities"].sum()
    )

    assert set(exposures["Source"]).issubset(
        set(banks["Id"])
    )

    assert set(exposures["Target"]).issubset(
        set(banks["Id"])
    )

    print("Validation passed.")

In [10]:
# ============================================================
# 8. EXPORT GEPHI-COMPATIBLE CSV FILES
# ============================================================

def export_dataset(banks, exposures):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Nodes: Gephi recognizes Id and Label.
    banks.to_csv(
        OUTPUT_DIR / "banks.csv",
        index=False,
        float_format="%.10f"
    )

    # Edges: Gephi recognizes Source, Target, Type and Weight.
    exposures.to_csv(
        OUTPUT_DIR / "exposures.csv",
        index=False,
        float_format="%.10f"
    )

    metadata = pd.DataFrame([{
        "number_of_banks": NUM_BANKS,
        "number_of_edges": len(exposures),
        "seed": SEED,
        "power_law_exponent": POWER_LAW_EXPONENT,
        "attachment_edges": ATTACHMENT_EDGES,
        "customer_loan_recovery_rate": RECOVERY_RATE,
        "total_interbank_exposure": exposures["Weight"].sum(),
        "source_paper": "Krause and Giansante (2012)"
    }])

    metadata.to_csv(
        OUTPUT_DIR / "network_metadata.csv",
        index=False
    )

    print("\nDataset generated successfully.")
    print(f"Banks: {len(banks)}")
    print(f"Directed edges: {len(exposures)}")

    print("\nFiles:")
    print(OUTPUT_DIR / "banks.csv")
    print(OUTPUT_DIR / "exposures.csv")
    print(OUTPUT_DIR / "network_metadata.csv")

In [ ]:
# Generate bank sizes
assets = generate_bank_sizes(
    NUM_BANKS,
    POWER_LAW_EXPONENT,
    MIN_ASSETS,
    MAX_ASSETS
)

# Generate initial balance sheets
banks = generate_balance_sheets(assets)

# Generate directed interbank network
G = generate_network(assets)

# Allocate interbank loan exposures
exposures = allocate_exposures(
    G,
    banks
)

# Rebalance balance sheets
banks = rebalance_balance_sheets(
    banks,
    exposures
)

# Add network metrics
banks = add_network_metrics(
    banks,
    exposures
)

# Validate the generated dataset
validate_dataset(
    banks,
    exposures
)

# Export CSV files
export_dataset(
    banks,
    exposures
)

Validation passed.

Dataset generated successfully.
Banks: 200
Directed edges: 600

Files:
data\banks.csv
data\exposures.csv
data\network_metadata.csv
